## Exercise: Recommender systems

#### In this exercise, we will build a content-based recommendation system using a dataset of Netflix titles. We will preprocess the text data, convert it into numerical features with TF-IDF, and compute item similarities to generate recommendations. This hands-on activity will help use understand and implement key techniques in content-based filtering.

### **Learning Objectives**
#### By the end of this exercise, you should be able to:
#### - Understand content-based recommendation systems.
#### - Clean and preprocess text data.
#### - Convert text data into numerical features using TF-IDF.
#### - Compute item similarities using cosine similarity.
#### - Build and evaluate a content-based recommendation model.

In [1]:
import numpy as np
import pandas as pd

import re
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import linear_kernel

In [2]:
data = pd.read_csv('https://raw.githubusercontent.com/Explore-AI/Public-Data/master/Data/unsupervised_sprint/netflix_titles.csv', index_col=0)
data.head()

,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
show_id,,,,,,,,,,,
81145628,Movie,Norm of the North: King Sized Adventure,"Richard Finn, Tim Maltby","Alan Marriott, Andrew Toth, Brian Dobson, Cole...","United States, India, South Korea, China","September 9, 2019",2019,TV-PG,90 min,"Children & Family Movies, Comedies",Before planning an awesome wedding for his gra...
80117401,Movie,Jandino: Whatever it Takes,NaN,Jandino Asporaat,United Kingdom,"September 9, 2016",2016,TV-MA,94 min,Stand-Up Comedy,Jandino Asporaat riffs on the challenges of ra...
70234439,TV Show,Transformers Prime,NaN,"Peter Cullen, Sumalee Montano, Frank Welker, J...",United States,"September 8, 2018",2013,TV-Y7-FV,1 Season,Kids' TV,"With the help of three human allies, the Autob..."
80058654,TV Show,Transformers: Robots in Disguise,NaN,"Will Friedle, Darren Criss, Constance Zimmer, ...",United States,"September 8, 2018",2016,TV-Y7,1 Season,Kids' TV,When a prison ship crash unleashes hundreds of...
80125979,Movie,#realityhigh,Fernando Lebrija,"Nesta Cooper, Kate Walsh, John Michael Higgins...",United States,"September 8, 2017",2017,TV-14,99 min,Comedies,When nerdy high schooler Dani finally attracts...


### **Exercises**

#### In this exercise, we focus on the relevant columns `cast`, `title`, `description`, and `listed_in` because these textual features provide detailed descriptions and attributes essential for capturing the content similarities between media items. These columns contain detailed information about what the media is about, who stars in it, and its genres, which are crucial for generating meaningful recommendations in a content-based filtering approach.

## **Exercise 1: Data cleaning and preprocessing**
#### Before proceeding with our recommender system, we need to clean and process our data first to get the most accurate results.
#### We need to do the following:
#### - Remove rows with missing or NaN values.
#### - Remove punctuation and extra space in the text data. This helps to standardize and clean the text, ensuring consistency in the dataset and facilitating accurate analysis and modelling by eliminating unnecessary noise and variations in the text.

### **Hint**
#### - For all the text columns, remove all characters that are not alphanumeric or whitespace.
#### - For the 'cast' column, first remove all spaces and then replace commas with spaces. This ensures that the cast members' names are treated as single entities separated by spaces.
#### - Combine the columns `listed_in`, `cast`, `title`, and `description` into a single feature for the recommendation system. This creates a richer and more complete representation of each item, enhancing the effectiveness of the recommendation system by allowing it to reconsider all aspects of the content simultaneously. **Hint:** Remember to drop the individual columns as they are now combined into one.
#### - Drop the rest of the columns to streamline and focus on the most relevant data for our recommendation model so that we are only left with the `type`, `title`, and `combined` columns with `type` and `title` providing context and identification, and `combined` serving as the main feature for calculating similarities.

In [3]:
data.dtypes

type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

##### Algorithm steps
##### 1. Remove rows with missing or NaN values
##### 2. Remove punctuation and extra spaces in the text data
#####    - Remove characters that are not alphanumeric/whitespace
#####    - Remove all spaces from the case column and then replace commas with spaces
##### 3. Combine columns listed_in, cast, title, and description into a single feature
##### 4. Drop other columns to remain with type, title, and combined columns

In [4]:
data.dropna(inplace=True)

In [5]:
data.isna().any()

type            False
title           False
director        False
cast            False
country         False
date_added      False
release_year    False
rating          False
duration        False
listed_in       False
description     False
dtype: bool

In [6]:
text_cols = data.select_dtypes(include=['object', 'string']).columns

data[text_cols] = data[text_cols].replace(r'[^a-zA-Z0-9\s]', '', regex=True)

In [7]:
data['cast'] = data['cast'].str.replace(' ', '', regex=False).str.replace(',', ' ', regex=False)


In [8]:
data.iloc[0]['cast']

'AlanMarriottAndrewTothBrianDobsonColeHowardJenniferCameronJonathanHolmesLeeTockarLisaDuruptMayaKayMichaelDobson'

In [9]:
cols = ['listed_in', 'cast', 'title', 'description']

data['combined_features'] = data[cols].apply(' '.join, axis=1)

data.head()

,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,combined_features
show_id,,,,,,,,,,,,
81145628,Movie,Norm of the North King Sized Adventure,Richard Finn Tim Maltby,AlanMarriottAndrewTothBrianDobsonColeHowardJen...,United States India South Korea China,September 9 2019,2019,TVPG,90 min,Children Family Movies Comedies,Before planning an awesome wedding for his gra...,Children Family Movies Comedies AlanMarriottA...
80125979,Movie,realityhigh,Fernando Lebrija,NestaCooperKateWalshJohnMichaelHigginsKeithPow...,United States,September 8 2017,2017,TV14,99 min,Comedies,When nerdy high schooler Dani finally attracts...,Comedies NestaCooperKateWalshJohnMichaelHiggin...
70304989,Movie,Automata,Gabe Ibez,AntonioBanderasDylanMcDermottMelanieGriffithBi...,Bulgaria United States Spain Canada,September 8 2017,2014,R,110 min,International Movies SciFi Fantasy Thrillers,In a dystopian future an insurance adjuster fo...,International Movies SciFi Fantasy Thrillers ...
80164077,Movie,Fabrizio Copano Solo pienso en mi,Rodrigo Toro Francisco Schultz,FabrizioCopano,Chile,September 8 2017,2017,TVMA,60 min,StandUp Comedy,Fabrizio Copano takes audience participation t...,StandUp Comedy FabrizioCopano Fabrizio Copano ...
70304990,Movie,Good People,Henrik Ruben Genz,JamesFrancoKateHudsonTomWilkinsonOmarSySamSpru...,United States United Kingdom Denmark Sweden,September 8 2017,2014,R,90 min,Action Adventure Thrillers,A struggling couple cant believe their luck wh...,Action Adventure Thrillers JamesFrancoKateHud...


In [10]:
data.iloc[0]

type                                                             Movie
title                           Norm of the North King Sized Adventure
director                                       Richard Finn Tim Maltby
cast                 AlanMarriottAndrewTothBrianDobsonColeHowardJen...
country                          United States India South Korea China
date_added                                            September 9 2019
release_year                                                      2019
rating                                                            TVPG
duration                                                        90 min
listed_in                             Children  Family Movies Comedies
description          Before planning an awesome wedding for his gra...
combined_features    Children  Family Movies Comedies AlanMarriottA...
Name: 81145628, dtype: object

In [11]:
data.iloc[0]['combined_features']

'Children  Family Movies Comedies AlanMarriottAndrewTothBrianDobsonColeHowardJenniferCameronJonathanHolmesLeeTockarLisaDuruptMayaKayMichaelDobson Norm of the North King Sized Adventure Before planning an awesome wedding for his grandfather a polar bear king must take back a stolen artifact from an evil archaeologist first'

In [12]:
keep_cols = ['type', 'title', 'combined_features']

data.drop(columns=[col for col in data.columns if col not in keep_cols], inplace=True)

data.head()

,type,title,combined_features
show_id,,,
81145628,Movie,Norm of the North King Sized Adventure,Children Family Movies Comedies AlanMarriottA...
80125979,Movie,realityhigh,Comedies NestaCooperKateWalshJohnMichaelHiggin...
70304989,Movie,Automata,International Movies SciFi Fantasy Thrillers ...
80164077,Movie,Fabrizio Copano Solo pienso en mi,StandUp Comedy FabrizioCopano Fabrizio Copano ...
70304990,Movie,Good People,Action Adventure Thrillers JamesFrancoKateHud...


In [13]:
data.shape

(3774, 3)

## **Exercise 2: Feature extraction**

#### Next, we want to convert the combined text feature into numerical features using TF-IDF. This enables the application of mathematical and statistical techniques for measuring similarities between different media items. In its raw form, text data cannot be directly used for similarity calculations or machine learning algorithms. By transforming the text into numerical representations, we can leverage these techniques to analyse and compare the content effectively.

#### - Utilise TF-IDF to convert the `combined` column into numerical vectors, which represent the importance of words in the document. Initialise your TF-IDF vectorizer without specifying any parameters, which means it will default to single-word tokens.
#### - Compute the cosine similarity between these vectors to measure how similar the titles are.

In [14]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(data['combined_features'])
cosine_sim = linear_kernel(tfidf_matrix)

## **Exercise 3: Building the recommendation function**

#### Now, we can generate recommendations based on cosine similarity.
#### Define a function that, given a title, finds similar titles by looking up their cosine similarity scores and returns the top 10 recommendations based on these scores.

In [25]:
def get_similar_titles(title, df, cosine_sim, n=10):
    matches = df.index[df['title'] == title]

    if len(matches) == 0:
        return f"Title '{title} not found.'"

    idx = df.index.get_loc(matches[0])

    scores = list(enumerate(cosine_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    scores = [(i, score) for i, score in scores if i != idx][:n]

    return pd.DataFrame({
        'title': [df.iloc[i]['title'] for i, _ in scores],
        'similarity': [score for _, score in scores]
    })

In [26]:
get_similar_titles('realityhigh', data, cosine_sim)

,title,similarity
0,Follow Me,0.140659
1,The Only Mother To You All,0.129911
2,The World Is Yours,0.123456
3,Deadcon,0.122376
4,The Walls of the Moon,0.119250
5,Friend Request,0.113360
6,Christmas Crush,0.112264
7,Brick,0.111463
8,Gnome Alone,0.108188
9,Sadies Last Days on Earth,0.104728


In [27]:
data.shape

(3774, 3)

### **Exercise 4: Test the recommender**

#### Say you are trying to get recommendations for what movie to watch, and you particularly enjoyed the film `The Crown`. Run our recommender for this title and see what recommendations we get.

#### Would you want to watch any of these titles?

In [28]:
get_similar_titles('The Crown', data, cosine_sim)

"Title 'The Crown not found.'"

In [31]:
'The Crown' in data['title']

False